In [1]:
import pandas as pd
import numpy as np

In [59]:
events = pd.read_csv("../../../data/processed/bookclubs_seattle_clean.csv")
events.head()

,title,link,description,when,day_of_week_start,day_of_week_end,start_date,end_date,start_time,end_time,...,address,city_state,venue_name,venue_rating,venue_reviews,venue_search_link,thumbnail,book_title,book_author,tags
0,Silent Book Club Night!!,https://www.eventbrite.com/e/silent-book-club-...,"Join a diverse community that loves reading, m...","Thu, Feb 26, 6:30 – 8:30 PM",Thu,Thu,2026-02-26,2026-02-26,6:30 PM,8:30 PM,...,"Quinn's Pub, 1001 E Pike St","Seattle, WA",Quinn's Pub,4.3,748.0,https://www.google.com/search?sca_esv=529d083b...,https://encrypted-tbn0.gstatic.com/images?q=tb...,NaN,NaN,[]
1,Bookish Trivia Night,https://theticket.seattletimes.com/calendar/?_...,Join Silent Book Club Capitol Hill and Freeze ...,"Thu, Feb 19, 6:30 – 9:00 PM",Thu,Thu,2026-02-19,2026-02-19,6:30 PM,9:00 PM,...,"a/stir, 818 E Pike St","Seattle, WA",a/stir,4.4,2769.0,https://www.google.com/search?sca_esv=529d083b...,https://encrypted-tbn0.gstatic.com/images?q=tb...,NaN,NaN,"['romance', 'trivia']"
2,Book Club - The Sword of Kaigen by M.L. Wang,https://www.meetup.com/fremontfantasy/,Welcome to the Fremont Fantasy Book Club! We w...,"Sun, Feb 22, 12:30 – 2:00 PM",Sun,Sun,2026-02-22,2026-02-22,12:30 PM,2:00 PM,...,"Fremont Branch - The Seattle Public Library, 7...","Seattle, WA",Fremont Branch - The Seattle Public Library,4.6,80.0,https://www.google.com/search?sca_esv=529d083b...,https://www.google.com/maps/vt/data=QLd0KA20JQ...,The Sword of Kaigen,M.L. Wang,['fantasy']
3,Obec Book Club — Ballard Brewed Coalition,https://www.ballardbrewed.com/events/nhq3ga7q0...,Obec Book Club The book for February will be s...,"Mon, Feb 23, 6 – 8 PM",Mon,Mon,2026-02-23,2026-02-23,6:00 PM,8:00 PM,...,"Obec Brewing, 1144 NW 52nd St","Seattle, WA",Obec Brewing,4.6,327.0,https://www.google.com/search?sca_esv=529d083b...,https://www.google.com/maps/vt/data=tjrCzKF3jw...,NaN,NaN,[]
4,Asylum Book Club,https://www.eventbrite.com/e/silent-reading-ja...,"Unwind with a book, global jazz, and bottomles...","Tue, Feb 24, 6 – 8 PM",Tue,Tue,2026-02-24,2026-02-24,6:00 PM,8:00 PM,...,"Asylum, 108 S Jackson St ste b","Seattle, WA",Asylum,4.2,18.0,https://www.google.com/search?sca_esv=529d083b...,https://encrypted-tbn0.gstatic.com/images?q=tb...,Night @ Asylum | 7PM–9PM 📍 Asylum Collective,Asylum,[]


In [74]:
import ast
from datetime import datetime
import pandas as pd


# --- Lightweight helpers for simple recommendations ---

def _parse_tags(value):
    """Convert a stringified list like "['romance']" to a Python list."""
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []


def _normalize_tags_column(events: pd.DataFrame) -> pd.DataFrame:
    events = events.copy()
    def _norm(val):
        tags = _parse_tags(val)
        return [str(t).strip().lower() for t in tags if str(t).strip()]
    events["tags"] = events["tags"].apply(_norm)
    return events


def load_events(csv_path="../../../data/processed/bookclubs_seattle_clean.csv"):
    events = pd.read_csv(csv_path)
    # Parse tags and datetimes
    events["tags"] = events["tags"].apply(_parse_tags)
    events["start_iso"] = pd.to_datetime(events["start_iso"], errors="coerce")
    events["end_iso"] = pd.to_datetime(events["end_iso"], errors="coerce")
    events = _normalize_tags_column(events)
    return events


def filter_upcoming(events: pd.DataFrame, now=None):
    now = now or pd.Timestamp.now()
    if not pd.api.types.is_datetime64_any_dtype(events.get("start_iso")):
        events = events.copy()
        events["start_iso"] = pd.to_datetime(events["start_iso"], errors="coerce")
    return events.loc[events["start_iso"] >= now].copy()


def _recency_bonus(start_ts, now):
    """Favor next ~2 weeks strongly; taper, then penalize far future."""
    if pd.isna(start_ts):
        return 0
    days = max((start_ts - now).total_seconds() / 86400.0, 0)
    # Boost up to 14 days, taper to 0 by ~45 days, then penalize
    if days <= 14:
        return 3.0 - 0.15 * days  # ~0.9 at 14d
    if days <= 45:
        return 0.9 - 0.03 * (days - 14)  # down to ~0 at 45d
    return -0.05 * (days - 45)  # push farther future down


def score_event(row, user_tags: set[str], preferred_city: str | None, now=None):
    now = now or pd.Timestamp.now()
    base_score = 0.5  # baseline so non-overlap items can surface

    event_tags = set(row.get("tags", []))
    tag_overlap = len(event_tags & user_tags)
    tag_score = min(3, tag_overlap)  # cap tag influence

    recency_score = _recency_bonus(row.get("start_iso"), now)

    city_score = 0
    if preferred_city and isinstance(row.get("city_state"), str):
        if preferred_city.lower() in row["city_state"].lower():
            city_score = 2

    venue_score = 0
    vr = row.get("venue_rating")
    if pd.notna(vr):
        try:
            venue_score = 0.2 * float(vr)
        except Exception:
            pass

    # Weighting favors recency first, then location, then tags
    # Emphasis on recency (moderate), then city, then tags
    score = base_score + 1.5 * recency_score + 1.0 * city_score + 0.75 * tag_score + 0.2 * venue_score
    return score, tag_overlap, tag_score, recency_score, city_score, venue_score


def recommend(
    events: pd.DataFrame,
    user_tags: list[str],
    top_k: int = 10,
    preferred_city: str | None = None,
    now=None,
):
    now = now or pd.Timestamp.now()
    events = _normalize_tags_column(events)
    user_tag_set = set(t.strip().lower() for t in user_tags)
    upcoming = filter_upcoming(events, now=now)
    upcoming[[
        "score",
        "tag_overlap",
        "tag_score",
        "recency_score",
        "city_score",
        "venue_score",
    ]] = upcoming.apply(
        lambda r: pd.Series(score_event(r, user_tag_set, preferred_city, now=now)), axis=1
    )
    ranked = upcoming.sort_values(
        ["recency_score", "city_score", "tag_overlap", "score", "start_iso"],
        ascending=[False, False, False, False, True],
    )

    # Promote some exploration: inject low-overlap items every 3 slots
    explore_pool = ranked[ranked["tag_overlap"] == 0]
    main_pool = ranked[ranked["tag_overlap"] > 0]

    results = []
    explore_iter = iter(explore_pool.itertuples(index=False))
    seen = set()

    for idx, row in enumerate(main_pool.itertuples(index=False), start=1):
        if len(results) >= top_k:
            break
        if row.link in seen:
            continue
        results.append(row)
        seen.add(row.link)
        if idx % 3 == 0 and len(results) < top_k:
            try:
                e = next(explore_iter)
                if e.link not in seen:
                    results.append(e)
                    seen.add(e.link)
            except StopIteration:
                pass

    # If we still need items, pull from remaining explore then from ranked
    if len(results) < top_k:
        for e in explore_pool.itertuples(index=False):
            if len(results) >= top_k:
                break
            if e.link not in seen:
                results.append(e)
                seen.add(e.link)
    if len(results) < top_k:
        for r in ranked.itertuples(index=False):
            if len(results) >= top_k:
                break
            if r.link not in seen:
                results.append(r)
                seen.add(r.link)

    def _has_trivia(row) -> bool:
        tags = getattr(row, "tags", [])
        return any(str(t).lower() == "trivia" for t in tags)

    # Ensure the first slot is not a trivia event when possible
    if results and _has_trivia(results[0]):
        for idx in range(1, len(results)):
            if not _has_trivia(results[idx]):
                results[0], results[idx] = results[idx], results[0]
                break

    # Final deterministic ordering after exploration/trivia swap
    df_out = pd.DataFrame(results)
    # Final ordering: highest cumulative score first; break ties by earlier start
    df_out = df_out.sort_values(["score", "start_iso"], ascending=[False, True])

    def _has_trivia_tags(val) -> bool:
        tags = val
        if isinstance(tags, str):
            try:
                tags = ast.literal_eval(tags)
            except Exception:
                tags = []
        return any(str(t).lower() == "trivia" for t in (tags or []))

    # Enforce: first result should not be trivia when possible
    if not df_out.empty and _has_trivia_tags(df_out.iloc[0]["tags"]):
        for idx in range(1, len(df_out)):
            if not _has_trivia_tags(df_out.iloc[idx]["tags"]):
                order = [idx] + list(range(0, idx)) + list(range(idx + 1, len(df_out)))
                df_out = df_out.iloc[order].reset_index(drop=True)
                break

    out_cols = [
        "title",
        "start_iso",
        "end_iso",
        "city_state",
        "venue_name",
        "book_title",
        "book_author",
        "tags",
        "link",
        "score",
        "tag_overlap",
        "tag_score",
        "recency_score",
        "city_score",
        "venue_score",
    ]
    return df_out[out_cols].head(top_k)



In [77]:
user_prefs = ["lgbtq"]
preferred_city = "Seattle"
recommend(events, user_prefs, preferred_city=preferred_city, top_k=5)

,title,start_iso,end_iso,city_state,venue_name,book_title,book_author,tags,link,score,tag_overlap,tag_score,recency_score,city_score,venue_score
3,Heated Rivalry Book Club,2026-02-19 18:00:00,2026-02-19T19:30:00,"Burien, WA",Page 2 Books,NaN,NaN,[],https://www.eventbrite.com/e/heated-rivalry-bo...,5.175501,0.0,0.0,2.989001,0.0,0.96
1,"March Book Club - ""Spear"" by Nicola Griffith",2026-03-14 21:30:00,2026-03-14T23:30:00,"Seattle, WA",NaN,Spear,Nicola Griffith,"[lgbtq, literature]",https://www.meetup.com/lesbianlit-117/events/3...,4.185138,1.0,1.0,0.623425,2.0,0.00
4,Queer Book Club,2026-04-01 19:00:00,NaN,"Seattle, WA",El Sueñito Brewing Company,NaN,NaN,[lgbtq],https://do206.com/events/2026/4/1/queer-book-c...,3.559825,1.0,1.0,0.086550,2.0,0.90
0,Read and Rant: The South Sound's LGBTQ+ Book Club,2026-03-14 11:00:00,2026-03-14T13:00:00,"Tacoma, WA",Tacoma Public Library Moore Branch,NaN,NaN,"[lgbtq, literature, nonfiction, young adult]",https://tacoma.bibliocommons.com/events/688514...,2.392825,1.0,1.0,0.636550,0.0,0.94
2,Engayging in Sapphistry: A Book Club,2026-03-15 14:00:00,2026-03-15T15:00:00,"Redmond, WA",Redmond Library,NaN,NaN,"[classics, lgbtq]",https://kcls.bibliocommons.com/events/68f00af1...,2.338200,1.0,1.0,0.602800,0.0,0.92
